In [0]:
dbutils.widgets.text("catalog", "claims")
catalog = dbutils.widgets.get("catalog")
# R6 bundle: raw reference data stays in the shared SOURCE catalog (never duplicated per target).
# Defaults to the target catalog for backward compatibility (claims_daily passes nothing -> claims).
dbutils.widgets.text("source_catalog", catalog)
source_catalog = dbutils.widgets.get("source_catalog")

In [0]:
from pyspark.sql.functions import *

#claims_enriched

In [0]:
# Defensive drop so the create() path below rebuilds claims_enriched from scratch each run.
# (Was hardcoded to claims.gold; now follows the catalog widget — required for the R6
# bundle's dev/prod targets.)
spark.sql(f"DROP TABLE IF EXISTS {catalog}.gold.claims_enriched")

In [0]:
claims_src = spark.read.table(f"{catalog}.silver.claims_clean")
dim_policy = spark.read.table(f"{catalog}.silver.dim_policy_scd2")
# Raw reference data stays in the shared source catalog (R6: raw is never duplicated per target)
plans     = spark.read.table(f"{source_catalog}.raw.plans")
hospitals = spark.read.table(f"{source_catalog}.raw.hospitals")

# As-of plan lookup: the dim_policy_scd2 version valid on admission_date
plan_as_of_admission = (claims_src
    .select("policy_id", "admission_date").distinct().alias("c")
    .join(dim_policy.alias("d"),
          (col("c.policy_id") == col("d.policy_id")) &
          (col("d.__START_AT") <= col("c.admission_date")) &
          (col("d.__END_AT").isNull() | (col("c.admission_date") < col("d.__END_AT"))),
          "left")
    .select(
        col("c.policy_id"),
        col("c.admission_date"),
        col("d.plan_code").alias("plan_code_at_admission")))

# Policy inception: earliest effective date across the policy's full history
policy_inception = (dim_policy
    .groupBy("policy_id")
    .agg(min("__START_AT").alias("policy_start_date")))

network_history = (spark.read.format("csv")
    .option("header", "true")
    .schema("hospital_id STRING, network_from DATE, network_to DATE")
    .load("/Volumes/claims/raw/landing/hospital_network_history.csv"))

enriched = (claims_src
    .join(plan_as_of_admission, ["policy_id", "admission_date"], "left")
    .join(policy_inception, "policy_id", "left")
    .join(plans.select(
            col("plan_code").alias("plan_code_at_admission"),
            col("plan_family"),
            col("room_rent_cap_per_day")),
          "plan_code_at_admission", "left")
    .join(hospitals.select("hospital_id", "city_tier"), "hospital_id", "left"))

network_flags = (claims_src.select("claim_id", "hospital_id", "admission_date")
    .join(network_history, "hospital_id", "left")
    .withColumn("in_network_on_date",
        col("network_from").isNotNull() &
        (col("network_from") <= col("admission_date")) &
        (col("network_to").isNull() | (col("admission_date") <= col("network_to"))))
    .groupBy("claim_id")
    .agg(max("in_network_on_date").alias("is_network_at_admission")))

# R5 deliverable: exactly 30 columns, no more, no fewer
claims_enriched_df = (enriched
    .join(network_flags, "claim_id", "left")
    .select(
        col("claim_id"),
        col("member_id"),
        col("policy_id"),
        col("hospital_id"),
        col("admission_date"),
        col("discharge_date"),
        date_trunc("month", col("admission_date")).cast("date").alias("admission_month"),
        col("diagnosis_code"),
        col("procedure_code"),
        col("is_pre_existing"),
        col("claim_type"),
        col("billed_amount"),
        col("room_rent_per_day"),
        col("room_days"),
        col("status"),
        col("approved_amount"),
        col("rejection_code"),
        col("submitted_ts"),
        col("settled_ts"),
        col("channel"),
        col("plan_code_at_admission"),
        col("plan_family"),
        col("room_rent_cap_per_day"),
        col("policy_start_date"),
        floor(months_between(col("admission_date"), col("policy_start_date"))).cast("int").alias("policy_tenure_months"),
        datediff(col("discharge_date"), col("admission_date")).cast("int").alias("length_of_stay"),
        (col("room_rent_per_day") > col("room_rent_cap_per_day")).alias("room_rent_over_cap"),
        when(col("settled_ts").isNotNull(),
             datediff(col("settled_ts"), col("submitted_ts")).cast("int")).alias("turnaround_days"),
        coalesce(col("is_network_at_admission"), lit(False)).alias("is_network_at_admission"),
        col("city_tier").alias("hospital_city_tier"),
    ))

# R7: claims_enriched must be clustered by admission_month, hospital_id.
if spark.catalog.tableExists(f"{catalog}.gold.claims_enriched"):
    spark.sql(f"ALTER TABLE {catalog}.gold.claims_enriched "
              f"CLUSTER BY (admission_month, hospital_id)")
else:
    (claims_enriched_df.writeTo(f"{catalog}.gold.claims_enriched")
        .using("delta")
        .clusterBy("admission_month", "hospital_id")
        .create())

#hospital_monthly

In [0]:
claims_gold = spark.read.table(f"{catalog}.gold.claims_enriched")

hospital_monthly_df = (claims_gold
    .groupBy("hospital_id", "admission_month")
    .agg(
        count("*").cast("int").alias("claim_count"),
        round(sum("billed_amount"), 2).cast("decimal(18,2)").alias("total_billed"),
        coalesce(round(sum("approved_amount"), 2), lit(0.0))
            .cast("decimal(18,2)").alias("total_approved"),
        sum(when(col("status") == "rejected", 1).otherwise(0))
            .cast("int").alias("rejected_count")))

# Idempotent publish: overwrite on every run (claims_daily job runs this daily).
(hospital_monthly_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(f"{catalog}.gold.hospital_monthly"))

# R2 documentation is schema-level metadata; re-apply it after every overwrite
spark.sql(f"COMMENT ON TABLE {catalog}.gold.hospital_monthly IS 'Gold monthly aggregate per hospital and admission month: claim counts, billed/approved totals and rejected counts.'")
monthly_comments = {
    "hospital_id": "Hospital identifier",
    "admission_month": "First-of-month bucket of admission_date",
    "claim_count": "Number of claims for the hospital-month",
    "total_billed": "Sum of billed_amount for the hospital-month",
    "total_approved": "Sum of approved_amount for the hospital-month, coalesced to 0 when none approved",
    "rejected_count": "Number of claims with status = rejected",
}
for mcol, cmt in monthly_comments.items():
    spark.sql(f"ALTER TABLE {catalog}.gold.hospital_monthly ALTER COLUMN {mcol} COMMENT '{cmt}'")

display(spark.sql(f"SELECT count(*) AS hospital_monthly_rows FROM {catalog}.gold.hospital_monthly"))

#R7

In [0]:
#R7
history = spark.sql(f"DESCRIBE HISTORY {catalog}.gold.claims_enriched")
display(history.select("version", "timestamp", "operation",
                       "operationParameters", "operationMetrics"))

# --- 2) Time travel: current vs the first version ever written ---
print("rows at current version:",
      spark.sql(f"SELECT count(*) FROM {catalog}.gold.claims_enriched").first()[0])
try:
    print("rows at VERSION AS OF 0 (as first created):",
          spark.sql(f"SELECT count(*) FROM {catalog}.gold.claims_enriched VERSION AS OF 0").first()[0])
except Exception as e:
    # R1 adds a row filter (claims_state_filter) on this table; Delta does not support
    # time travel on tables with row-level security policies, so skip in job runs.
    print("Time travel skipped — unsupported on a table with a row filter:", str(e)[:200])

# --- 3) Clustering metadata + physical layout ---
detail = spark.sql(f"DESCRIBE DETAIL {catalog}.gold.claims_enriched").collect()[0]
print("clusteringColumns:", detail["clusteringColumns"])
print("numFiles:", detail["numFiles"])

In [0]:
print("hospital_monthly rows:",
      spark.sql(f"SELECT count(*) FROM {catalog}.gold.hospital_monthly").first()[0])